# CRISP-DM — Fases 2 e 3: Análise Exploratória e Preparação dos Dados

**Projeto:** Mapa de Calor e Predição do Risco de Acidentes de Trânsito na Rodovia BR-230  
**Disciplina:** Ciência de Dados — PPGCC / UFMA  
**Equipe:** João Batista Santos Silva Neto, Ricardo Diniz Mendes, Winy Gabrielly Soares de Oliveira  
**Metodologia:** CRISP-DM (Fase 2: Entendimento e Análise Exploratória; Fase 3: Preparação dos Dados)  
**Recorte Temporal e Espacial:** Quadriênio pós-pandemia **2022–2025**, rodovia **BR-230** (Transamazônica)

Este notebook consolida formalmente o fluxo integrado de Análise Exploratória de Dados (EDA) e Engenharia/Preparação de Dados segundo o padrão CRISP-DM e as convenções da biblioteca `pandas` (*Data Wrangling Cheat Sheet*). Todas as grandezas, tabelas de contingência e gráficos são derivados de dados reais da Polícia Rodoviária Federal (PRF).


## 1. Objetivos da integração entre Análise Exploratória e Preparação

Este notebook unifica as diretrizes conceituais da **Aula 4 (Fase 2 — Entendimento dos Dados)** com a esteira de engenharia da **Fase 3 (Preparação dos Dados)**:

1. **Taxonomia Formal das Variáveis (Aula 4):** Identificação e validação das escalas de medição (Stevens, 1946) para cada atributo real da PRF, delimitando as operações matemáticas e estatísticas admitidas.
2. **Exploração Estatística Univariada e Bivariada (Aula 4):** Sumarização empírica de tendência central (média, mediana), dispersão (desvio padrão, IQR), forma da distribuição (assimetria, curtose) e quantificação de *outliers* univariados via amplitude interquartil (Tukey, 1.5 IQR).
3. **Limpeza e Tratamento da Qualidade:** Tratamento de valores sentinelas (`0` numéricos em quilometragem), decomposição de traçados compostos, identificação de inconsistências cadastrais e ajuste de marcos da via segundo a malha oficial do DNIT/SNV.
4. **Integração Relacional e Engenharia de Atributos:** Junção (*merge* 1:N) com a base granular de pessoas/veículos, sumarizando indicadores pré-sinistro de motocicletas, pedestres e idade média.
5. **Codificação Categórica e Matrizes Supervisionadas:** Aplicação de One-Hot Encoding nas variáveis nominais com preservação de colunas entre treino/teste e imputação de valores faltantes restrita ao conjunto de treino para prevenir vazamento de dados (*data leakage*).
6. **Integração Geoespacial e Persistência:** Geração de geometrias vetoriais WGS84 (`Point`, EPSG:4326) para subsidiar o mapa de calor por Kernel Density Estimation (KDE) e gravação dos artefatos em `bases/preparadas/`.


## 2. Ambiente, caminhos e carga reprodutível


In [ ]:
from pathlib import Path
import io
import subprocess
import sys
import tempfile
import zipfile
import warnings
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 140)

plt.rcParams.update({
    "figure.figsize": (11, 4.6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
    "axes.titlesize": 13,
})


def localizar_raiz() -> Path:
    aqui = Path.cwd().resolve()
    for candidato in [aqui, aqui.parent]:
        if (candidato / "bases").exists() and (candidato / "entendimento_dados").exists():
            return candidato
    return aqui


RAIZ = localizar_raiz()
BASES = RAIZ / "bases"
PREPARADAS = BASES / "preparadas"
FIGURAS = RAIZ / "preparacao_dos_dados" / "figuras"
PREPARADAS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

PRF_DOWNLOAD_LINKS = {
    "datatran2025.csv": {"id": "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR", "ano": 2025},
    "datatran2024.csv": {"id": "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO", "ano": 2024},
    "datatran2023.csv": {"id": "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq", "ano": 2023},
    "datatran2022.csv": {"id": "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-", "ano": 2022},
    "acidentes2025.csv": {"id": "1-Gp9S-ALO0D1nT8S_OKoC8xlW7BY8F82", "ano": 2025},
    "acidentes2024.csv": {"id": "14lVfqdoE2gxDliaKZu7K9Mx6847maPtl", "ano": 2024},
    "acidentes2023.csv": {"id": "1-Yk6TV00CH3PixTkKmkoUJQsNiUc5xLm", "ano": 2023},
    "acidentes2022.csv": {"id": "1sd8YIqIHapYaxQeaa-6yeWMdVLTl6hgD", "ano": 2022},
}


def bundle_autoridades_ssl() -> str | bool:
    """Junta certifi às CAs do chaveiro do macOS (proxy corporativo / Netskope)."""
    try:
        import certifi
        base = Path(certifi.where()).read_text()
        fallback = certifi.where()
    except Exception:
        return True
    extras = ""
    if sys.platform == "darwin":
        try:
            extras = subprocess.check_output(
                ["security", "find-certificate", "-a", "-p", "/Library/Keychains/System.keychain"],
                text=True,
                timeout=15,
            )
        except Exception:
            extras = ""
    if "BEGIN CERTIFICATE" not in extras:
        return fallback
    destino = Path(tempfile.gettempdir()) / "cienciadados-ca-bundle.pem"
    destino.write_text(f"{base}\n{extras}")
    return str(destino)


def garantir_base(nome: str, pasta: Path = BASES) -> Path:
    destino = pasta / nome
    if destino.exists():
        return destino
    info = PRF_DOWNLOAD_LINKS.get(nome)
    if not info:
        raise FileNotFoundError(f"{nome} ausente em {pasta} e sem link configurado.")
    file_id = info["id"]
    print(f"Baixando {nome} (ID: {file_id})...")
    session = requests.Session()
    session.verify = bundle_autoridades_ssl()
    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    r = session.get(url, stream=True, timeout=120)
    if r.status_code != 200 or len(r.content) < 1000:
        url_alt = f"https://docs.google.com/uc?export=download&id={file_id}"
        r = session.get(url_alt, stream=True, timeout=120)
    r.raise_for_status()
    destino.write_bytes(r.content)
    print(f"Salvo em {destino} ({destino.stat().st_size / 1024**2:.1f} MB)")
    return destino


def ler_prf(nome: str) -> pd.DataFrame:
    caminho = garantir_base(nome)
    return pd.read_csv(
        caminho,
        sep=";",
        encoding="latin-1",
        decimal=",",
        low_memory=False,
    )


## 3. Ingestão e Formatação dos Dados Oficiais (PRF 2022–2025)

A PRF publica os dados originais com codificação `ISO-8859-1`, separador de campo ponto-e-vírgula (`;`) e vírgula como separador decimal. Realizamos o carregamento seguro, a tipagem temporal de datas e horas e a padronização das colunas para minúsculas, filtrando as ocorrências da **BR-230**.


In [ ]:
ANOS = [2022, 2023, 2024, 2025]
EXTENSAO_DNIT = {"PB": 521.0, "TO": 144.7}


def formatar_ocorrencias(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    out["ano_arquivo"] = ano
    if "data_inversa" in out.columns:
        out["data"] = pd.to_datetime(out["data_inversa"], errors="coerce")
        out["ano"] = out["data"].dt.year.fillna(ano).astype(int)
        out["mes"] = out["data"].dt.month
        out["dia"] = out["data"].dt.day
    if "horario" in out.columns:
        out["hora"] = pd.to_datetime(
            out["horario"].astype(str), format="%H:%M:%S", errors="coerce"
        ).dt.hour
    for col_num in ["km", "latitude", "longitude"]:
        if col_num in out.columns:
            out[col_num] = pd.to_numeric(out[col_num], errors="coerce")
    for col_int in ["pessoas", "mortos", "feridos_leves", "feridos_graves", "ilesos", "ignorados", "feridos", "veiculos"]:
        if col_int in out.columns:
            out[col_int] = pd.to_numeric(out[col_int], errors="coerce").fillna(0).astype(int)
    return out


def carregar_ocorrencias(anos=ANOS, rodovia=230) -> pd.DataFrame:
    partes = []
    for ano in anos:
        bruto = ler_prf(f"datatran{ano}.csv")
        fmt = formatar_ocorrencias(bruto, ano)
        # Seleção estrita da BR-230 (suporta numérico ou texto)
        filtro_br = pd.to_numeric(fmt["br"], errors="coerce").eq(rodovia)
        recorte_ano = fmt[filtro_br].copy()
        print(f"{ano}: nacional={len(fmt):,} | BR-230={len(recorte_ano):,}")
        partes.append(recorte_ano)
    return pd.concat(partes, ignore_index=True)


br230_bruta = carregar_ocorrencias()
print(f"\nConsolidado BR-230: {len(br230_bruta):,} acidentes | "
      f"{br230_bruta['id'].nunique():,} IDs únicos | "
      f"período {br230_bruta['data'].min().date()} → {br230_bruta['data'].max().date()}")
display(
    br230_bruta.groupby("ano")
    .agg(acidentes=("id", "size"), mortos=("mortos", "sum"), feridos=("feridos", "sum"))
    .astype({"mortos": int, "feridos": int})
)


## 4. Taxonomia e Escalas de Medida dos Atributos (Aula 4)

De acordo com a **Aula 4 (Slides 20–27)**, a identificação formal da escala de medição de cada variável é pré-requisito indispensável para validar a legitimidade estatística das operações analíticas e transformações subsequentes. Construímos a taxonomia a partir dos dados reais da BR-230 conforme a teoria clássica de Stevens (1946):

- **Nominal:** Relações de igualdade/diferença ($=, \neq$). Estatística admissível: Moda e frequências relativas.
- **Ordinal:** Relações de ordem ($>, <$). Estatística admissível: Mediana, percentis.
- **Intervalar:** Escala com zero arbitrário. Operações de adição/subtração admitidas ($+,-$).
- **Razão:** Escala com zero absoluto e natural. Operações aritméticas completas admitidas ($+,-,\times,/$) e cálculo legítimo de média aritmética, desvio padrão e coeficientes de variação.


In [ ]:
def construir_taxonomia_dados_reais(df: pd.DataFrame) -> pd.DataFrame:
    mapeamento_stevens = {
        "id": ("Nominal", "=, !=", "Moda", "Chave primária do acidente"),
        "data": ("Intervalar", "=, !=, >, <, +", "Mediana, Média", "Data do sinistro"),
        "horario": ("Intervalar", "=, !=, >, <, +", "Mediana, Média", "Hora do sinistro"),
        "dia_semana": ("Nominal / Ordinal", "=, !=, >, <", "Moda, Mediana", "Dia da semana"),
        "uf": ("Nominal", "=, !=", "Moda", "Unidade Federativa"),
        "br": ("Nominal", "=, !=", "Moda", "Identificador da rodovia"),
        "km": ("Razão", "=, !=, >, <, +, -, *, /", "Média, Desvio Padrão", "Quilômetro do sinistro"),
        "municipio": ("Nominal", "=, !=", "Moda", "Município de ocorrência"),
        "causa_acidente": ("Nominal", "=, !=", "Moda", "Causa principal apontada"),
        "tipo_acidente": ("Nominal", "=, !=", "Moda", "Natureza do impacto"),
        "classificacao_acidente": ("Ordinal", "=, !=, >, <", "Mediana, Moda", "Severidade do desfecho"),
        "fase_dia": ("Nominal / Ordinal", "=, !=, >, <", "Moda", "Luminosidade natural"),
        "sentido_via": ("Nominal", "=, !=", "Moda", "Sentido do fluxo"),
        "condicao_metereologica": ("Nominal", "=, !=", "Moda", "Condição meteorológica"),
        "tipo_pista": ("Nominal", "=, !=", "Moda", "Configuração física da pista"),
        "tracado_via": ("Nominal", "=, !=", "Moda", "Geometria da rodovia"),
        "uso_solo": ("Nominal (Binária)", "=, !=", "Moda", "Ambiente urbano vs rural"),
        "pessoas": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Total de pessoas envolvidas"),
        "mortos": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Vítimas fatais"),
        "feridos_leves": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Feridos leves"),
        "feridos_graves": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Feridos graves"),
        "ilesos": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Indivíduos ilesos"),
        "ignorados": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Estado físico ignorado"),
        "feridos": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Total de feridos"),
        "veiculos": ("Razão (Discreta)", "=, !=, >, <, +, -, *, /", "Média, Mediana", "Veículos envolvidos"),
        "latitude": ("Intervalar", "=, !=, >, <, +", "Média, Mediana", "Coordenada geográfica"),
        "longitude": ("Intervalar", "=, !=, >, <, +", "Média, Mediana", "Coordenada geográfica"),
    }
    
    linhas = []
    n_total = len(df)
    for col in df.columns:
        if col in mapeamento_stevens:
            escala, operacoes, medidas, papel = mapeamento_stevens[col]
            n_dist = df[col].nunique(dropna=True)
            n_nulos = df[col].isna().sum()
            exemplo = str(df[col].dropna().iloc[0]) if not df[col].dropna().empty else "-"
            linhas.append({
                "Atributo": col,
                "Escala (Stevens)": escala,
                "Operações Admissíveis": operacoes,
                "Estatísticas Apropriadas": medidas,
                "Tipo / Papel": papel,
                "Valores Distintos": n_dist,
                "Nulos": n_nulos,
                "Exemplo Real": exemplo[:25]
            })
    return pd.DataFrame(linhas)

tab_taxonomia = construir_taxonomia_dados_reais(br230_bruta)
display(tab_taxonomia)


## 5. Análise Exploratória dos Dados (EDA — Aula 4)

Conforme a **Aula 4**, calculamos para cada atributo métrico relevante as seguintes grandezas estatísticas essenciais:

- **Medidas de Tendência Central:** Média aritmética e Mediana (Slide 27).
- **Medidas de Dispersão:** Desvio padrão e Amplitude Interquartil ($IQR = Q_3 - Q_1$) (Slides 28 e 34).
- **Medidas de Forma:** Coeficiente de Assimetria (*skewness*) e Curtose (*kurtosis*) (Slides 29–30).
- **Detecção Numérica de Outliers:** Regra de Tukey com barreiras a $1{,}5 \times IQR$ (Slides 35–37).


In [ ]:
def sumario_estatistico_aula4(df: pd.DataFrame, colunas: list = None) -> pd.DataFrame:
    if colunas is None:
        candidatas = ["km", "hora", "pessoas", "veiculos", "feridos", "mortos"]
        colunas = [c for c in candidatas if c in df.columns]

    resumo = []
    for col in colunas:
        s = pd.to_numeric(df[col], errors="coerce").dropna()
        n = len(s)
        if n == 0:
            continue
        q1 = s.quantile(0.25)
        med = s.median()
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lim_inf = q1 - 1.5 * iqr
        lim_sup = q3 + 1.5 * iqr
        outliers = s[(s < lim_inf) | (s > lim_sup)].count()

        media = s.mean()
        desvio = s.std()
        assimetria = s.skew()
        curtose = s.kurtosis()

        if assimetria > 0.5:
            formato_dist = "Assimétrica à Direita (Positiva)"
        elif assimetria < -0.5:
            formato_dist = "Assimétrica à Esquerda (Negativa)"
        else:
            formato_dist = "Aproximadamente Simétrica"

        resumo.append({
            "Atributo": col,
            "N": n,
            "Média": round(media, 2),
            "Mediana": round(med, 2),
            "Desvio Padrão": round(desvio, 2),
            "Q1": round(q1, 2),
            "Q3": round(q3, 2),
            "IQR": round(iqr, 2),
            "Lim. Inf. (Tukey)": round(lim_inf, 2),
            "Lim. Sup. (Tukey)": round(lim_sup, 2),
            "Outliers (1.5 IQR)": outliers,
            "Assimetria": round(assimetria, 2),
            "Curtose": round(curtose, 2),
            "Forma da Distribuição": formato_dist
        })
    return pd.DataFrame(resumo)

tab_eda = sumario_estatistico_aula4(br230_bruta)
display(tab_eda)


### 5.1 Análise Bivariada e Distribuições Conjuntas de Risco (Aula 4, Slide 31)

Analisamos a distribuição conjunta dos fatores contextuais reais frente à ocorrência de óbitos. Criamos uma coluna indicadora temporária `alvo_fatal_temp` baseada na classificação oficial da PRF para mensurar taxas empíricas de letalidade por categoria.


In [ ]:
br230_bruta["alvo_fatal_temp"] = (br230_bruta["classificacao_acidente"] == "Com Vítimas Fatais").astype(int)

def tabela_contingencia_risco(df: pd.DataFrame, coluna_cat: str) -> pd.DataFrame:
    tab = df.groupby(coluna_cat, observed=False).agg(
        Total_Acidentes=("id", "count"),
        Fatais=("alvo_fatal_temp", "sum"),
        Taxa_Fatalidade_Pct=("alvo_fatal_temp", lambda x: round(100 * x.mean(), 2)),
        Total_Mortos=("mortos", "sum")
    ).reset_index()
    tab = tab.sort_values(by="Total_Acidentes", ascending=False)
    return tab

print("Risco Bivariado por Tipo de Pista:")
display(tabela_contingencia_risco(br230_bruta, "tipo_pista"))

print("\nRisco Bivariado por Fase do Dia:")
display(tabela_contingencia_risco(br230_bruta, "fase_dia"))


### 5.2 Visualizações Gráficas da Exploração (Aula 4, Slides 32 e 37)

Geramos as representações gráficas canônicas recomendadas na Aula 4, derivadas estritamente da base real da BR-230:
1. **Subplot A:** Boxplots de dispersão de quilometragem por UF (Slide 37);
2. **Subplot B:** Histograma de frequências horárias e Ogiva Acumulada de Galton (Slide 32) com calibração dinâmica do teto vertical para impedir sobreposição entre as barras e os títulos;
3. **Subplot C:** Mapa de calor bivariado de risco (Tipo de Pista $\times$ Fase do Dia).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), dpi=150)

# Subplot A: Boxplots de km por UF ordenados dinamicamente pelo volume real
ufs_ordem = br230_bruta["uf"].value_counts().index.tolist()
dados_km_uf = [br230_bruta.query("uf == @u")["km"].dropna().values for u in ufs_ordem]
labels_uf = [f"{u}\n(n={len(br230_bruta.query('uf == @u'))})" for u in ufs_ordem]

bplot = axes[0].boxplot(dados_km_uf, patch_artist=True, tick_labels=labels_uf, medianprops=dict(color="#b42828", linewidth=2))
for patch in bplot["boxes"]:
    patch.set_facecolor("#c8d7eb")
    patch.set_edgecolor("#143264")
axes[0].set_title("A. Dispersão de Km por UF (Boxplots)", fontsize=11, fontweight="bold", pad=10)
axes[0].set_ylabel("Quilômetro na BR-230 (km)")
axes[0].set_ylim(0, 1600)
axes[0].tick_params(axis="x", labelsize=7.5)

# Subplot B: Histograma e Ogiva Acumulada Horária com calibração dinâmica
contagem_hora = br230_bruta["hora"].value_counts().sort_index()
horas = contagem_hora.index
freq_abs = contagem_hora.values
ogiva = 100 * np.cumsum(freq_abs) / len(br230_bruta)

pico_max = freq_abs.max()
# Teto com folga generosa para garantir isolamento vertical total do título
teto_y = int(math.ceil(pico_max * 1.35 / 100.0) * 100)

bars = axes[1].bar(horas, freq_abs, color="#c8d7eb", edgecolor="#143264", label="Acidentes (Absoluto)")
axes[1].set_title("B. Frequência Horária e Ogiva Acumulada", fontsize=11, fontweight="bold", pad=10)
axes[1].set_xlabel("Hora do Dia (0 a 23h)")
axes[1].set_ylabel("Frequência Absoluta (Acidentes)")
axes[1].set_ylim(0, teto_y)

# Destacar valor de pico dinamicamente via caixa anotada sem colisão
idx_pico = np.argmax(freq_abs)
h_pico = horas[idx_pico]
v_pico = freq_abs[idx_pico]
axes[1].annotate(
    f"Pico: {v_pico} (18h)",
    xy=(h_pico, v_pico),
    xytext=(h_pico - 6.0, v_pico + 75),
    arrowprops=dict(arrowstyle="->", color="#143264", lw=1.5),
    fontsize=8.5,
    fontweight="bold",
    color="#143264",
    bbox=dict(boxstyle="round,pad=0.25", fc="#ffffff", ec="#143264", alpha=0.9)
)

ax_ogiva = axes[1].twinx()
ax_ogiva.plot(horas, ogiva, color="#b42828", marker="o", markersize=4, label="Ogiva Acumulada (%)")
ax_ogiva.set_ylabel("Frequência Acumulada (%)", color="#b42828")
ax_ogiva.set_ylim(0, 105)

# Subplot C: Heatmap de Letalidade Bivariada (Pista x Fase do Dia)
crosstab_mat = pd.crosstab(
    br230_bruta["fase_dia"],
    br230_bruta["tipo_pista"],
    values=br230_bruta["alvo_fatal_temp"],
    aggfunc="mean"
) * 100

ordem_fases = [f for f in ["Pleno dia", "Plena Noite", "Amanhecer", "Anoitecer"] if f in crosstab_mat.index]
ordem_pistas = [p for p in ["Simples", "Dupla", "Múltipla"] if p in crosstab_mat.columns]
crosstab_mat = crosstab_mat.reindex(index=ordem_fases, columns=ordem_pistas)
crosstab_tot = pd.crosstab(br230_bruta["fase_dia"], br230_bruta["tipo_pista"]).reindex(index=ordem_fases, columns=ordem_pistas)

im = axes[2].imshow(crosstab_mat.values, cmap="YlOrRd", aspect="auto")
axes[2].set_xticks(range(len(crosstab_mat.columns)))
axes[2].set_yticks(range(len(crosstab_mat.index)))
axes[2].set_xticklabels(crosstab_mat.columns)
axes[2].set_yticklabels(crosstab_mat.index)
axes[2].set_title("C. Letalidade Bivariada: Pista × Iluminação (%)", fontsize=11, fontweight="bold", pad=10)

for i in range(len(crosstab_mat.index)):
    for j in range(len(crosstab_mat.columns)):
        val = crosstab_mat.iloc[i, j]
        tot = crosstab_tot.iloc[i, j]
        txt = f"{val:.1f}%\n(n={tot})" if not np.isnan(val) else "N/A"
        axes[2].text(j, i, txt, ha="center", va="center", color="black" if (val < 15 or np.isnan(val)) else "white", fontweight="bold", fontsize=9)

fig.tight_layout()
fig.savefig(FIGURAS / "exploracao_distribuicoes_outliers.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura EDA salva em:", FIGURAS / "exploracao_distribuicoes_outliers.png")


## 6. Limpeza dos Dados e Rastreio da Qualidade (Aula 4 & Fase 3)

Respondendo sistematicamente aos problemas de qualidade diagnosticados:

1. **Traçados Compostos:** Registros com múltiplos traçados separados por ponto-e-vírgula (`Reta;Curva`) são decompostos retendo o primeiro elemento em `tracado_principal` e sinalizando o traçado composto na flag booleana `tracado_composto`.
2. **Uso do Solo e Ambiente:** Normalização textual dos valores para derivação das variáveis `area_urbana` e `area_rural`.
3. **Divergências Administrativas:** Identificação de divergências entre a UF territorial do município e a regional operacional da PRF na flag `uf_diverge_regional`.
4. **Alinhamento com Malha DNIT:** No trecho tocantinense (TO), quilometragens excedentes à extensão física do Sistema Nacional de Viação (teto de 144,7 km) são ajustadas em `km_ajustado` e rastreadas por `km_acima_dnit`.


In [ ]:
def limpar_ocorrencias(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Variável-alvo (desfecho)
    out["fatal"] = out["classificacao_acidente"].eq("Com Vítimas Fatais")
    out["alvo_fatal"] = out["fatal"].astype(int)

    # Traçado composto: "Reta;Curva" → primeiro valor + flag
    tracado = out["tracado_via"].fillna("").astype(str).str.strip()
    out["tracado_composto"] = tracado.str.contains(";", regex=False)
    out["tracado_principal"] = (
        tracado.str.split(";")
        .str[0]
        .str.strip()
        .replace({"": np.nan})
    )

    # Uso do solo: Sim=Urbano, Não=Rural
    uso = out["uso_solo"].astype(str).str.strip().str.lower()
    out["area_urbana"] = uso.eq("sim")
    out["area_rural"] = uso.eq("não")

    # Divergência regional vs município
    reg_uf = out["regional"].astype(str).str.extract(r"([A-Z]{2})$")[0]
    out["uf_regional"] = reg_uf
    out["uf_diverge_regional"] = out["uf"].ne(out["uf_regional"]) & out["uf_regional"].notna()

    # Quilômetro acima do cadastro DNIT (TO)
    limite = out["uf"].map(EXTENSAO_DNIT)
    out["km_acima_dnit"] = limite.notna() & (out["km"] > limite)
    out["km_ajustado"] = out["km"]
    mask_to = out["uf"].eq("TO") & out["km_acima_dnit"]
    out.loc[mask_to, "km_ajustado"] = EXTENSAO_DNIT["TO"]

    # Fim de semana e período noturno
    dia_clean = out["dia_semana"].astype(str).str.lower().str.strip()
    out["dia_semana_norm"] = dia_clean
    out["fim_de_semana"] = dia_clean.isin(["sexta-feira", "sábado", "sabado", "domingo"])

    fase = out["fase_dia"].astype(str).str.strip().str.lower()
    out["periodo_noturno"] = fase.isin(["plena noite", "anoitecer", "amanhecer"])

    # Combinações contextuais de risco
    out["rural_pista_simples"] = out["area_rural"] & out["tipo_pista"].eq("Simples")
    out["fora_pb"] = out["uf"].ne("PB")

    return out


br230 = limpar_ocorrencias(br230_bruta)

resumo_limpeza = pd.DataFrame([
    {"checagem": "Acidentes", "valor": len(br230)},
    {"checagem": "Fatais (alvo=1)", "valor": int(br230["alvo_fatal"].sum())},
    {"checagem": "Proporção fatal (%)", "valor": round(100 * br230["alvo_fatal"].mean(), 2)},
    {"checagem": "Traçados compostos", "valor": int(br230["tracado_composto"].sum())},
    {"checagem": "UF diverge da regional", "valor": int(br230["uf_diverge_regional"].sum())},
    {"checagem": "km acima do DNIT (TO/PB)", "valor": int(br230["km_acima_dnit"].sum())},
    {"checagem": "Rural + pista simples", "valor": int(br230["rural_pista_simples"].sum())},
    {"checagem": "Rural + simples fora PB", "valor": int((br230["rural_pista_simples"] & br230["fora_pb"]).sum())},
])
display(resumo_limpeza)

print("\nTraçado principal (top 8):")
display(br230["tracado_principal"].value_counts().head(8).rename("acidentes").to_frame())

print("Registros com km acima do DNIT:")
display(
    br230.loc[br230["km_acima_dnit"], ["id", "uf", "km", "km_ajustado", "municipio"]]
    .sort_values("km", ascending=False)
    .head(10)
)


## 7. Integração relacional com a base de pessoas

O arquivo original `acidentesYYYY.csv` tem granularidade de **1 linha = 1 pessoa envolvida**.
Para preservar a unidade de análise de **1 linha = 1 acidente** na base de modelagem preditiva, agregamos os dados por `id` de acidente, computando:

- `envolve_moto`: booleano indicando presença de motocicleta, motoneta ou ciclomotor;
- `envolve_pedestre`: booleano indicando atropelamento de pedestre;
- `idade_media`: média aritmética da idade das pessoas envolvidas no sinistro.


In [ ]:
def formatar_pessoas(df: pd.DataFrame, ano: int) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.strip().lower() for c in out.columns]
    out["ano_arquivo"] = ano
    if "idade" in out.columns:
        out["idade"] = pd.to_numeric(out["idade"], errors="coerce")
    for col in ["tipo_envolvido", "tipo_veiculo", "sexo", "estado_fisico"]:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip()
    return out


def carregar_pessoas(anos=ANOS, rodovia=230) -> pd.DataFrame:
    partes = []
    for ano in anos:
        bruto = ler_prf(f"acidentes{ano}.csv")
        fmt = formatar_pessoas(bruto, ano)
        filtro_br = pd.to_numeric(fmt["br"], errors="coerce").eq(rodovia)
        recorte = fmt[filtro_br].copy()
        partes.append(recorte)
        print(f"Pessoas {ano}: total={len(fmt):,} | BR-230={len(recorte):,}")
    return pd.concat(partes, ignore_index=True)


pessoas = carregar_pessoas()

# Agregação por acidente
agregado_pessoas = (
    pessoas.groupby("id")
    .agg(
        envolve_moto=("tipo_veiculo", lambda s: s.str.lower().str.contains("moto|ciclomotor|motoneta").any()),
        envolve_pedestre=("tipo_envolvido", lambda s: s.str.lower().str.contains("pedestre").any()),
        idade_media=("idade", "mean"),
    )
    .reset_index()
)
agregado_pessoas["envolve_moto"] = agregado_pessoas["envolve_moto"].fillna(False).astype(bool)
agregado_pessoas["envolve_pedestre"] = agregado_pessoas["envolve_pedestre"].fillna(False).astype(bool)

br230_int = br230.merge(agregado_pessoas, on="id", how="left")
br230_int["envolve_moto"] = br230_int["envolve_moto"].fillna(False)
br230_int["envolve_pedestre"] = br230_int["envolve_pedestre"].fillna(False)

print(f"Acidentes após merge: {len(br230_int):,} (esperado: {len(br230):,})")
print(f"Idade média preenchida em: {br230_int['idade_media'].notna().mean():.1%}")
print(f"Envolve moto: {br230_int['envolve_moto'].sum():,} ({br230_int['envolve_moto'].mean():.1%})")
print(f"Envolve pedestre: {br230_int['envolve_pedestre'].sum():,} ({br230_int['envolve_pedestre'].mean():.1%})")


## 8. Seleção de atributos para modelagem supervisionada

Isolamos estritamente os atributos contextuais pré-sinistro, evitando vazamento de dados (*data leakage*):
- **Excluídos do preditor:** Contagens de desfecho (`mortos`, `feridos`, `ilesos`), a classificação original do desfecho e atributos post-hoc (causas periciais).
- **Mantidos no preditor:** Características físicas e espaciais da rodovia, luminosidade, meteorologia, temporalidade e agregados de envolvimento de vulneráveis.


In [ ]:
COLS_ID = ["id", "ano", "data", "data_inversa", "horario"]

COLS_CONTEXTO = [
    "uf", "municipio", "km", "km_ajustado", "latitude", "longitude",
    "fase_dia", "periodo_noturno", "dia_semana_norm", "fim_de_semana",
    "tipo_pista", "tracado_principal", "uso_solo", "area_urbana", "area_rural",
    "condicao_metereologica", "sentido_via",
    "envolve_moto", "envolve_pedestre", "idade_media",
]

COLS_ALVO = ["alvo_fatal"]

COLS_SELECIONADAS = COLS_ID + COLS_CONTEXTO + COLS_ALVO
modelo = br230_int[COLS_SELECIONADAS].copy()

print(f"Matriz de modelagem: {modelo.shape[0]:,} linhas × {modelo.shape[1]} colunas")
print("\nValores ausentes nos preditores:")
nulos_contexto = modelo[COLS_CONTEXTO].isna().sum()
display(nulos_contexto[nulos_contexto > 0].rename("nulos").to_frame())


## 9. Construção da partição prospectiva temporal

Em dados de sinistros de trânsito, o particionamento temporal cronológico é indispensável para replicar condições reais de aplicação:
- **Treino:** Anos 2022 a 2024 (janela histórica consolidada);
- **Teste:** Ano 2025 (horizonte prospectivo estrito para validação fora do tempo).


In [ ]:
treino = modelo.query("ano <= 2024").copy()
teste = modelo.query("ano == 2025").copy()

particao = pd.DataFrame([
    {
        "conjunto": "Treino (2022–2024)",
        "n": len(treino),
        "fatais": int(treino["alvo_fatal"].sum()),
        "prop_fatal_%": round(100 * treino["alvo_fatal"].mean(), 2),
    },
    {
        "conjunto": "Teste (2025)",
        "n": len(teste),
        "fatais": int(teste["alvo_fatal"].sum()),
        "prop_fatal_%": round(100 * teste["alvo_fatal"].mean(), 2),
    },
    {
        "conjunto": "Total BR-230",
        "n": len(modelo),
        "fatais": int(modelo["alvo_fatal"].sum()),
        "prop_fatal_%": round(100 * modelo["alvo_fatal"].mean(), 2),
    },
])
display(particao)

# Cruzamento rural × pista
cruz = (
    br230_int.assign(
        uso=np.where(br230_int["area_urbana"], "Urbano", "Rural"),
    )
    .groupby(["fora_pb", "uso", "tipo_pista"], dropna=False)
    .agg(acidentes=("id", "size"), prop_fatal=("alvo_fatal", "mean"), mortos=("mortos", "sum"))
    .reset_index()
)
cruz["recorte"] = np.where(cruz["fora_pb"], "Fora da PB", "PB")
display(
    cruz.query("uso == 'Rural' and tipo_pista == 'Simples'")
    .round({"prop_fatal": 3})
)


## 9.5. Encoding de Categóricas e Separação das Matrizes (X e y)

Para que os algoritmos de aprendizado de máquina (como Redes Neurais MLP, Regressão Logística e Random Forest) possam processar os dados adequadamente:
1. Transformamos variáveis categóricas nominais em colunas numéricas binárias (One-Hot Encoding) via `pd.get_dummies(..., drop_first=True)` para evitar colinearidade perfeita;
2. Realinhamos as colunas da partição de teste com `teste_encoded.reindex(columns=treino_encoded.columns, fill_value=0)` para garantir consistência dimensional estrita;
3. Tratamos os valores faltantes de `idade_media` calculando a mediana estritamente sobre a partição de treino e aplicando-a a ambos os conjuntos, eliminando qualquer risco de vazamento de dados (*data leakage*);
4. Isolamos explicitamente os preditores contextuais ($X$) da variável-alvo de letalidade ($y$).


In [ ]:
# Definir quais colunas categóricas nominais serão transformadas
cols_nominais = ["tipo_pista", "condicao_metereologica", "uso_solo", "fase_dia", "tracado_principal", "sentido_via"]

# Aplicar One-Hot Encoding (drop_first=True evita multicolinearidade)
treino_encoded = pd.get_dummies(treino, columns=cols_nominais, drop_first=True)
teste_encoded = pd.get_dummies(teste, columns=cols_nominais, drop_first=True)

# Garantir que as colunas do teste sejam exatamente iguais às do treino após o encoding
teste_encoded = teste_encoded.reindex(columns=treino_encoded.columns, fill_value=0)

# Imputação de nulos em idade_media baseada exclusivamente no treino (prevenção de vazamento)
mediana_idade_treino = treino_encoded["idade_media"].median()
treino_encoded["idade_media"] = treino_encoded["idade_media"].fillna(mediana_idade_treino)
teste_encoded["idade_media"] = teste_encoded["idade_media"].fillna(mediana_idade_treino)
print(f"Mediana de idade_media (treino): {mediana_idade_treino:.1f} anos (aplicada no treino e teste)")

# Lista final de preditores numéricos e booleanos
cols_excluir = ["id", "ano", "data", "data_inversa", "horario", "uf", "municipio", "alvo_fatal", "dia_semana_norm"]
features_finais = [c for c in treino_encoded.columns if c not in cols_excluir]

# Separação explícita de X (preditores) e y (alvo)
X_treino = treino_encoded[features_finais].astype(float)
y_treino = treino_encoded["alvo_fatal"].astype(int)

X_teste = teste_encoded[features_finais].astype(float)
y_teste = teste_encoded["alvo_fatal"].astype(int)

print(f"Dimensão X_treino: {X_treino.shape} | y_treino: {y_treino.shape}")
print(f"Dimensão X_teste:  {X_teste.shape} | y_teste: {y_teste.shape}")
print(f"Nulos em X_treino: {X_treino.isna().sum().sum()} | em X_teste: {X_teste.isna().sum().sum()}")


## 10. Visualizações de controle da preparação

Gráficos de controle do pipeline para validação visual da estabilidade temporal anual e da calibração do risco relativo por tipo de pista, salvos em alta definição (`dpi=150`).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), dpi=150)
plt.subplots_adjust(wspace=0.35)

# Volume e letalidade por ano com eixos independentes e espacados
por_ano = (
    br230_int.groupby("ano")
    .agg(acidentes=("id", "size"), prop_fatal=("alvo_fatal", "mean"))
    .reset_index()
)
axes[0].bar(por_ano["ano"].astype(str), por_ano["acidentes"], color="#2c5f8a", label="Acidentes", width=0.55)
axes[0].set_ylabel("Acidentes", color="#143264", fontweight="bold")
axes[0].set_title("A. Volume anual — BR-230 preparada", fontsize=11, fontweight="bold", pad=10)
axes[0].set_ylim(0, 2500)

for bar in axes[0].patches:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, h + 40, f"{int(h)}", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#143264")

ax2 = axes[0].twinx()
ax2.plot(por_ano["ano"].astype(str), 100 * por_ano["prop_fatal"], color="#b42828", marker="o", linewidth=2)
ax2.set_ylabel("Proporção fatal (%)", color="#b42828", fontweight="bold")
ax2.set_ylim(0, 16)

for x_val, y_val in zip(por_ano["ano"].astype(str), 100 * por_ano["prop_fatal"]):
    ax2.text(x_val, y_val + 0.5, f"{y_val:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold", color="#b42828")

# Preditores categóricos: pista x alvo
resumo_pista = (
    br230_int.groupby("tipo_pista")["alvo_fatal"]
    .agg(acidentes="size", prop="mean")
    .reindex(["Simples", "Dupla", "Múltipla"])
)
cores_pista = ["#b42828", "#2c5f8a", "#7a8699"]
bars2 = axes[1].bar(resumo_pista.index.astype(str), 100 * resumo_pista["prop"], color=cores_pista, width=0.5)
axes[1].set_ylabel("Proporção fatal (%)", color="#143264", fontweight="bold")
axes[1].set_title("B. Letalidade por tipo de pista", fontsize=11, fontweight="bold", pad=10)
axes[1].set_ylim(0, 18)

for bar, n_acid in zip(bars2, resumo_pista["acidentes"]):
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.4, f"{yval:.1f}%\n(n={n_acid:,})".replace(",", "."), ha="center", va="bottom", fontsize=9, fontweight="bold", color="#1e293b")

fig.savefig(FIGURAS / "preparacao_controle_anual_pista.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura de controle salva em:", FIGURAS / "preparacao_controle_anual_pista.png")


## 10.5. Integração Geoespacial

Para viabilizar a modelagem espacial de densidade de Kernel (Kernel Density Estimation — KDE) e mapas temáticos, os sinistros com coordenadas válidas são convertidos em geometrias vetoriais de pontos (`Point`) sob o sistema de coordenadas geodésicas WGS84 (EPSG:4326) utilizando GeoPandas e Shapely.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Filtrar registros que possuem coordenadas geográficas válidas
mask_geo = br230_int["latitude"].notna() & br230_int["longitude"].notna()
br230_geo = br230_int[mask_geo].copy()

# Criar a geometria vetorial de Pontos a partir de Longitude (x) e Latitude (y)
geometrias = [Point(xy) for xy in zip(br230_geo["longitude"], br230_geo["latitude"])]

# Converter para GeoDataFrame definindo o CRS WGS84 (EPSG:4326)
gdf_br230 = gpd.GeoDataFrame(br230_geo, geometry=geometrias, crs="EPSG:4326")

print(f"Registros geoespaciais criados: {len(gdf_br230):,} acidentes com geometria ativa.")
display(gdf_br230[["id", "uf", "municipio", "km", "geometry"]].head(3))


## 11. Persistência dos artefatos oficiais

Exportamos as matrizes preparadas e persistimos os artefatos oficiais em `bases/preparadas/`:
- `br230_ocorrencias_preparadas.csv`: Base consolidada com variáveis de engenharia e flags de auditoria;
- `br230_modelo_treino.csv` e `br230_modelo_teste.csv`: Partição prospectiva temporal;
- `br230_modelo_completo.csv`: Matriz contextual completa;
- `X_treino.csv`, `y_treino.csv`, `X_teste.csv`, `y_teste.csv`: Matrizes já codificadas e isoladas para treinamento supervisionado;
- `br230_espacial.geojson`: Camada espacial vetorial para estimação de Kernel (KDE).


In [ ]:
# Exportação padrão CSV
caminho_full = PREPARADAS / "br230_ocorrencias_preparadas.csv"
br230_int.to_csv(caminho_full, index=False)
treino.to_csv(PREPARADAS / "br230_modelo_treino.csv", index=False)
teste.to_csv(PREPARADAS / "br230_modelo_teste.csv", index=False)
modelo.to_csv(PREPARADAS / "br230_modelo_completo.csv", index=False)

# Exportação das matrizes separadas e codificadas
X_treino.to_csv(PREPARADAS / "X_treino.csv", index=False)
y_treino.to_csv(PREPARADAS / "y_treino.csv", index=False)
X_teste.to_csv(PREPARADAS / "X_teste.csv", index=False)
y_teste.to_csv(PREPARADAS / "y_teste.csv", index=False)

# Exportação do formato geoespacial (GeoJSON WGS84 EPSG:4326)
gdf_br230.to_file(PREPARADAS / "br230_espacial.geojson", driver="GeoJSON")

inventario = pd.DataFrame([
    {"arquivo": p.name, "tamanho_mb": round(p.stat().st_size / 1024**2, 2)}
    for p in sorted(PREPARADAS.glob("*.*")) if p.suffix in [".csv", ".geojson"]
])
display(inventario)


## 12. Síntese e Encaminhamentos para a Fase 4 (Modelagem)

1. **Universo Analítico Consolidado:** 6.217 acidentes da BR-230 no quadriênio pós-pandemia (2022–2025), com preenchimento completo de coordenadas geográficas e marcos quilométricos.
2. **Definição da Variável-Alvo:** `alvo_fatal = 1` se `classificacao_acidente == "Com Vítimas Fatais"`, representando 603 ocorrências fatais (taxa global de letalidade de 9,70%).
3. **Divisão Temporal Prospectiva:** Conjunto de Treino compreendendo 2022–2024 ($N = 4.472$; 435 fatais; taxa de 9,73%) e Conjunto de Teste compreendendo 2025 ($N = 1.745$; 168 fatais; taxa de 9,63%).
4. **Matrizes Numéricas de Modelagem:** One-Hot Encoding aplicado às nominais com tratamento de colunas não observadas (`reindex`) e imputação de `idade_media` pela mediana calculada no treino (39,0 anos), gerando matrizes $X_{\text{treino}}$ e $X_{\text{teste}}$ estritamente numéricas e livres de valores nulos.
5. **Base Geoespacial Ativa:** 6.217 feições vetoriais `Point` salvas em `br230_espacial.geojson` sob projeção WGS84 (EPSG:4326), aptas para o mapa de calor por KDE.
6. **Controle de Vazamento:** Remoção estrita das contagens de desfecho (`mortos`, `feridos`) dos preditores de entrada para garantir que o modelo aprenda exclusivamente a partir do contexto pré-sinistro da rodovia.
